# EOA - Protocolo Experimental em Notebook

Este *notebook* conduz a avaliação prática do framework **EOA (*Explanation-Oriented Agents*)** sobre os dados empíricos do sistema hidráulico da ZeMA.

- **Parte 1 (Demonstração Guiada):** roteiro estruturado contemplando a inicialização com persistência em SQLite (`eoa.db`), o acoplamento de ferramentas (*tools*) nos agentes CMA e AA, a execução no modo Zero-ML, a listagem de incidentes, a geração de explicações para saídas preditivas e a manipulação dinâmica de agentes especialistas.
- **Parte 2 (Experimentação Livre):** espaço aberto com sugestões de extensão para criação de novas células, instrumentação customizada e avaliação da experiência de desenvolvimento.

> Consulte [`GUIDE.md`](./GUIDE.md) para referência completa de assinaturas, parâmetros e contratos da API.

---
## Parte 1: Demonstração Guiada

### 1. Configuração do Ambiente
Carrega a chave da API do Gemini a partir do arquivo `.env` e define a função auxiliar responsável pela formatação das explicações, pontuações de auditoria e rastros de execução (*execution trace*).

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

# Carrega a chave de API definida no arquivo .env deste diretório
load_dotenv(Path(".env"))
print(f"GEMINI_API_KEY carregada: {bool(os.environ.get('GEMINI_API_KEY'))}")
print(f"GEMINI_API_KEY = {os.environ.get('GEMINI_API_KEY')}")


def mostrar_resposta(res) -> None:
    """Exibe a narrativa gerada, a pontuação do validador e o rastro dos agentes."""
    print("=" * 80)
    print("EXPLICAÇÃO GERADA:")
    print("-" * 80)
    print(res.narrative)
    print("-" * 80)
    print(
        f"Nota Validador: {res.validation_score}/10.0 | "
        f"Aprovado: {res.validation_passed} | "
        f"Tentativas: {res.attempts_used} | "
        f"Fallback: {res.fallback_used}"
    )
    print("-" * 80)
    for step in res.execution_trace:
        print(f"[{step.agent_name:<23}] {step.status:<10} | {step.execution_time_ms:7.1f} ms | {step.summary}")
    print("=" * 80)

GEMINI_API_KEY carregada: True


### 2. Importação dos Módulos e Inicialização com Persistência SQLite (`eoa.db`)
Importam-se os componentes públicos do EOA e as rotinas auxiliares de [`tools.py`](./tools.py). Em seguida, instancia-se `SQLiteSessionStore("eoa.db")` para persistência local e inicializa-se a camada `eoa` via construtor `ExplanationLayer`.

In [ ]:
from eoa import ExplanationLayer, SpecializedAgent, SQLiteSessionStore
from tools import (
    get_baseline_comparison,
    get_incident,
    get_sensor_readings,
    list_incidents,
    load_domain_context,
)

# Instancia o armazenamento SQLite local ('eoa.db') para sessões, histórico e logs de validação
store = SQLiteSessionStore("eoa.db")

# Inicializa a camada pelo construtor direto utilizando gemini-3.6-flash.
# O parâmetro auto_create_agents=True instancia automaticamente os quatro agentes padrão (CMA, AA, EA, VA).
eoa = ExplanationLayer(
    provider="gemini",
    model="gemini-3.6-flash",
    session_store=store,
    auto_create_agents=True,
)

eoa.inspect()

EOA Framework - Active Explanation Layer Configuration
Agent              Role                     Tools    Prompt Source
------------------------------------------------------------------------------
context_manager    Context Manager          0        Built-in
artifacts          Artifacts Analyst        0        Built-in
explainer          Explainer                0        Built-in
validator          Quality Gate Auditor     0        Built-in


### 3. Registro de Ferramentas no `ContextManagerAgent` (CMA)
Em substituição ao fornecimento manual de contexto a cada chamada, registram-se `load_domain_context`, `get_sensor_readings` e `get_baseline_comparison` como ferramentas executáveis pelo CMA, permitindo a recuperação autônoma da documentação técnica e das leituras dos sensores.

In [3]:
cma = eoa.get_agent("context_manager")

# load_domain_context(): carrega a documentação técnica da ZeMA e o dicionário dos 14 sensores
cma.register_tool(load_domain_context)

# get_sensor_readings(incident_id): retorna as médias de 60 segundos dos 14 sensores físicos do ciclo
cma.register_tool(get_sensor_readings)

# get_baseline_comparison(incident_id): compara o ciclo frente às estatísticas da operação saudável (100%)
cma.register_tool(get_baseline_comparison)

# Verifica na coluna 'Tools' o registro das três ferramentas no context_manager
eoa.inspect()

EOA Framework - Active Explanation Layer Configuration
Agent              Role                     Tools    Prompt Source
------------------------------------------------------------------------------
context_manager    Context Manager          3        Built-in
artifacts          Artifacts Analyst        0        Built-in
explainer          Explainer                0        Built-in
validator          Quality Gate Auditor     0        Built-in


### 4. Operação no Modo Zero-ML (Recuperação de Contexto via Ferramenta)
Invoca-se `eoa.explain(...)` fornecendo apenas a consulta operacional (`user_query`) e o identificador de sessão (`session_id`), sem injeção explícita de `context` ou artefatos preditivos. Nesse modo, o `ContextManagerAgent` aciona `load_domain_context` de forma autônoma, enquanto o `ArtifactsAgent` detecta a ausência de modelo preditivo e omite a etapa correspondente (`0.0 ms`).

In [4]:
eoa.clear_session("demo_zero_ml")

res_zero_ml = eoa.explain(
    session_id="demo_zero_ml",
    user_query=(
        "Explique em termos práticos como funciona o sistema hidráulico ZeMA, "
        "quais são os grupos de sensores físicos monitorados e o que significam "
        "os três códigos de condição do refrigerador (Cooler condition / %)."
    ),
)

mostrar_resposta(res_zero_ml)

EXPLICAÇÃO GERADA:
--------------------------------------------------------------------------------
No momento, não há documentação técnica, manuais operacionais ou artefatos de dados carregados no contexto do sistema referentes ao sistema hidráulico ZeMA, aos seus grupos de sensores físicos ou aos códigos percentuais de condição do refrigerador (*Cooler condition / %*).

Em conformidade com as diretrizes de rigor técnico e integridade das informações, não é possível detalhar o funcionamento específico, mapear os sensores ou definir os três níveis do refrigerador sem que os dados e esquemas correspondentes estejam integrados à base de conhecimento atual. 

Para que eu possa fornecer essa explicação detalhada e alinhada à sua operação, por favor, disponibilize os manuais do sistema, os schemas de dados ou os artefatos de monitoramento do ZeMA no sistema.
--------------------------------------------------------------------------------
Nota Validador: 10.0/10.0 | Aprovado: True | Tentativ

### 5. Inspeção dos Incidentes Disponíveis (`list_incidents`)
Antes de solicitar a explicação de uma predição específica, utiliza-se `list_incidents()` para inspecionar os nove incidentes representativos pré-processados com `RandomForestClassifier` e atribuições TreeSHAP.

In [5]:
# Guia de utilização das funções auxiliares de tools.py:
# - list_incidents()                     -> retorna os 9 incidentes selecionados (códigos 3, 20 e 100)
# - list_incidents(condition_code=3)     -> filtra apenas incidentes em falha crítica (3%)
# - load_domain_context()                -> carrega documentação do sistema ZeMA e catálogo de sensores
# - get_incident(incident_id)            -> retorna {'prediction': ..., 'xai': ...} do incidente especificado
# - get_sensor_readings(incident_id)     -> retorna as leituras médias dos 14 sensores físicos
# - get_baseline_comparison(incident_id) -> calcula delta, z-score e percentis vs. operação saudável (100%)

incidentes = list_incidents()

print(f"{'ID':<4} {'Ciclo':<7} {'Condição Prevista':<30} {'Conf.':<7} {'Stable':<7} {'Top 3 TreeSHAP'}")
print("-" * 100)
for inc in incidentes:
    top3 = ", ".join(f"{k}: {v:+.4f}" for k, v in inc["top_3_shap_features"].items())
    print(
        f"{inc['incident_id']:<4} {inc['cycle_id']:<7} "
        f"{inc['predicted_condition_label']:<30} {inc['confidence']:<7.2f} "
        f"{inc['stable_flag']:<7} {top3}"
    )

ID   Ciclo   Condição Prevista              Conf.   Stable  Top 3 TreeSHAP
----------------------------------------------------------------------------------------------------
1    2       3% (close to total failure)    0.50    1       FS2: +0.1935, PS4: +0.1051, PS6: -0.0762
2    241     3% (close to total failure)    1.00    0       PS6: +0.1670, PS5: +0.1557, TS4: +0.1340
3    492     3% (close to total failure)    1.00    0       PS6: +0.1588, PS5: +0.1482, TS4: +0.1174
4    731     3% (close to total failure)    0.98    0       PS6: +0.1815, PS5: +0.1574, TS4: +0.1327
5    737     20% (reduced efficiency)       0.58    1       VS1: +0.2067, PS5: -0.0599, PS3: +0.0444
6    1067    20% (reduced efficiency)       1.00    1       TS4: +0.1458, PS5: +0.1385, PS6: +0.1314
7    1458    20% (reduced efficiency)       1.00    0       TS4: +0.1347, PS5: +0.1302, PS6: +0.1277
8    1467    100% (full efficiency)         0.73    1       TS4: +0.2189, PS4: +0.0351, PS2: +0.0303
9    2202    100

### 6. Geração de Explicação para Predição em Modo Pull
Registra-se `get_incident` no `ArtifactsAgent` e informa-se `incident_id=2` (caso representativo de falha crítica `3%` em condição térmica estável). O orquestrador aciona automaticamente as ferramentas vinculadas aos agentes `artifacts` e `context_manager` para reunir predição, atribuições TreeSHAP, leituras dos sensores e desvios frente à linha de base saudável.

In [ ]:
# Vincula a ferramenta de recuperação de predição e XAI ao ArtifactsAgent
eoa.get_agent("artifacts").register_tool(get_incident)

eoa.clear_session("demo_incidente_2")

# Ao informar incident_id=2, AA e CMA recuperam predição, SHAP, contexto, telemetria e desvios via tools
res_predicao = eoa.explain(
    incident_id=2,
    session_id="demo_incidente_2",
    user_query=(
        "Por que o modelo classificou o incidente 2 como 3% (próximo à falha total)? "
        "Relacione os principais atributos do TreeSHAP com as leituras reais dos sensores "
        "e o quanto eles desviaram da operação saudável."
    ),
)

mostrar_resposta(res_predicao)

### 7. Ciclo de Vida de Agente Especialista: Criação, Sobrescrita de Prompt, Execução e Remoção
Demonstra-se a extensibilidade arquitetural mediante o registro de um `SpecializedAgent`, redefinição dinâmica de suas instruções via `override_agent_prompt`, execução aplicada ao Incidente 5 e remoção posterior com `remove_agent`.

In [ ]:
# 1. Instancia o agente especialista (na v0.1.0 o construtor requer um prompt inicial não vazio).
#    Inclui get_sensor_readings em tools=[] para obtenção paralela de stable_flag e TS1-TS4 na Fase 1.
agente_regime = SpecializedAgent(
    name="analista_regime_termico",
    role="Especialista em Estabilidade Térmica",
    prompt="Analise o incidente.",
    tools=[get_sensor_readings],
)
eoa.register_agent(agente_regime)

# 2. Sobrescreve o prompt do agente recém-criado referenciando as saídas das ferramentas ({tool_outputs})
eoa.override_agent_prompt(
    "analista_regime_termico",
    """Você é um Especialista em Estabilidade Térmica de Sistemas Hidráulicos.
Analise os dados de telemetria coletados ({tool_outputs}).
Em no máximo 3 linhas objetivas:
1. Informe se o ciclo está em regime estável (stable_flag=0) ou em transição (stable_flag=1).
2. Explique se os sensores de temperatura (TS1 a TS4) confirmam aquecimento consistente ou fase transitória.
""",
)

# Inspeciona a topologia atualizada e a origem do prompt ('Inline (str)')
eoa.inspect()

# 3. Executa uma consulta com o agente especialista integrado ao pipeline
eoa.clear_session("demo_incidente_5")
res_especialista = eoa.explain(
    incident_id=5,
    session_id="demo_incidente_5",
    user_query="Avalie a condição do refrigerador no incidente 5 considerando a estabilidade térmica do ciclo.",
)
mostrar_resposta(res_especialista)

# 4. Remove o agente especialista ao término da demonstração e verifica a restauração da topologia
eoa.remove_agent("analista_regime_termico")
eoa.inspect()

# Lista as sessões persistidas no banco SQLite 'eoa.db'
print("Sessões salvas em eoa.db:", store.list_sessions())

---
## Parte 2: Experimentação Livre (Agora é a sua vez)

A célula a seguir disponibiliza um espaço aberto para experimentação livre do framework EOA. Recomenda-se editar os trechos comentados, adicionar novas células ou propor variações alinhadas aos objetivos da avaliação.

Possibilidades de investigação (não exaustivas):
- **Incorporação de agentes especialistas:** instanciar `seu_agente = SpecializedAgent(...)` e registrá-lo via `eoa.register_agent(seu_agente)`.
- **Refinamento de instruções (*prompts*):** customizar agentes existentes com `eoa.override_agent_prompt("explainer", ...)` ou restaurar o padrão nativo via `eoa.reset_agent_prompt("explainer")`.
- **Acoplamento de novas ferramentas Python (*tools*):** implementar funções tipadas e vinculá-las por meio de `eoa.get_agent("context_manager").register_tool(sua_funcao)`.
- **Exploração de outros cenários:** consultar os demais incidentes (`1` a `9`), testar o modo Push (`prediction=` / `xai=`), conduzir diálogos multiturno com `session_id` (auditando o banco `eoa.db` via `store.get_validation_logs(...)`) ou desabilitar componentes específicos com `eoa.remove_agent(...)`.

In [ ]:
# Espaço aberto para experimentação livre do framework EOA (adicione novas células conforme necessário).
# Exemplos de referência para descomentário e adaptação:

# --- Sugestão 1: Criar e registrar um agente especialista customizado ---
# seu_agente = SpecializedAgent(
#     name="auditor_pressao",
#     role="Especialista em Pressão Hidráulica",
#     prompt="Analise exclusivamente os sensores PS1 a PS6 nos dados coletados ({tool_outputs}).",
#     tools=[get_sensor_readings],
# )
# eoa.register_agent(seu_agente)

# --- Sugestão 2: Implementar uma ferramenta Python e registrá-la no ContextManagerAgent ---
# def resumo_temperaturas(incident_id: int) -> dict:
#     """Retorna apenas as temperaturas TS1 a TS4 e seus desvios da linha de base."""
#     comp = get_baseline_comparison(incident_id)["comparisons"]
#     return {k: comp[k] for k in ("TS1", "TS2", "TS3", "TS4")}
#
# eoa.get_agent("context_manager").register_tool(resumo_temperaturas)

# --- Sugestão 3: Sobrescrever (e opcionalmente restaurar) o prompt do ExplainerAgent ---
# eoa.override_agent_prompt(
#     "explainer",
#     "Responda em 3 tópicos objetivos para operação técnica.\nEvidências: {artifacts}\nContexto: {domain_context}\nPergunta: {user_query}",
# )
# # eoa.reset_agent_prompt("explainer")

# --- Sugestão 4: Executar outro incidente (1 a 9), diálogo multiturno e auditoria no eoa.db ---
# res = eoa.explain(
#     incident_id=1,
#     session_id="meu_teste_01",
#     user_query="Por que a confiança do modelo ficou em apenas 50% no incidente 1?",
# )
# mostrar_resposta(res)
#
# res_continuacao = eoa.explain(
#     session_id="meu_teste_01",
#     user_query="Qual foi o valor exato do sensor FS2 nesse ciclo?",
# )
# mostrar_resposta(res_continuacao)
#
# print("Logs de validação no eoa.db:", store.get_validation_logs("meu_teste_01"))

eoa.inspect()